# Data Connectors from a Notebook

**Data · Beginner · ~10 min · pandas + fincept_connector · needs an internet connection**

Fincept Terminal ships with about 200 data connectors: small Python scripts in its `scripts` folder that fetch quotes, price history, FX rates, crypto, macro series and more. The terminal's screens use them, and so can your notebooks. `fincept_connector.run_enveloped` runs any connector and hands back one predictable result shape that drops straight into pandas. This notebook calls Yahoo Finance (quotes and price history), ExchangeRate (FX rates) and CoinPaprika (crypto), shows what a failed call looks like, and lists connectors that need no API key. It needs an internet connection and writes nothing to disk.

**What you will see**
- The envelope a connector returns, and how to turn it into a DataFrame
- A live quote table for several symbols and a six-month price chart
- Latest FX rates and the top crypto assets, from two keyless non-Yahoo sources
- A failed call (a missing API key) handled without an exception
- Keyless connectors from the terminal's catalogue, by category

## 1. Parameters

In [ ]:
# ── Parameters: edit these, then Run All (Ctrl+Shift+Enter) ──────────
QUOTE_SYMBOLS = ["AAPL", "MSFT", "NVDA", "SPY", "GLD"]   # any Yahoo Finance symbols
HISTORY_SYMBOL = "MSFT"      # one symbol for the price chart
HISTORY_DAYS = 180           # calendar days of history
FX_BASE = "USD"              # base currency for the FX table
FX_TARGETS = ["EUR", "GBP", "JPY", "INR", "CHF", "CNY"]
TOP_COINS = 10               # how many crypto assets to list

## 2. Imports and chart style

`fincept_connector` lives in the terminal's `scripts` folder, which is already on the notebook's Python path.

In [ ]:
import datetime as dt
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
import fincept_connector as fc

# One consistent, quiet chart style for every figure in this notebook
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "lines.linewidth": 1.6, "legend.frameon": False,
})

print("connectors are run from", os.path.dirname(os.path.abspath(fc.__file__)))

## 3. The envelope

`fc.run_enveloped([script, command, *args])` runs `scripts/<script>` exactly as the terminal does and always returns a dict of the same shape:

```
success:  {"ok": True,  "data": [{row}, {row}, ...], "columns": [...],
           "types": {column: "number" | "integer" | "string" | "date" | ...},
           "meta": {"source", "script", "command", "as_of", "count", "extra"}}
failure:  {"ok": False, "error": "plain message",
           "code": "missing_key" | "rate_limited" | "upstream_down" | "bad_args" | "no_data" | ...}
```

So every call follows the same recipe: check `ok`, then `pd.DataFrame(res["data"], columns=res["columns"])`. Two small helpers keep the rest of the notebook short:

- `run(argv)` calls `fc.run_enveloped` with an empty standard input. The Code tab's kernel has no terminal, and `run_enveloped` passes its own standard input on to the connector process.
- `fetch(script, command, *args)` returns the rows as a DataFrame, or prints the error and returns `None` when the call fails, so a failed fetch never stops the notebook.

Each call starts a separate Python process, so allow a second or two per call.

In [ ]:
def run(argv):
    """fc.run_enveloped(argv), giving the connector an empty standard input."""
    with open(os.devnull, encoding="utf-8") as empty:
        saved, sys.stdin = sys.stdin, empty
        try:
            return fc.run_enveloped(list(argv))
        finally:
            sys.stdin = saved


def fetch(*argv):
    """Run a connector; return its rows as a DataFrame, or None (after saying why) if it fails."""
    res = run(argv)
    if not res["ok"]:
        print(f"{' '.join(argv[:2])} failed [{res['code']}]: {res['error']}")
        return None
    if "tables" in res:      # a few connectors return several tables at once
        return {name: pd.DataFrame(t["rows"], columns=t["columns"]) for name, t in res["tables"].items()}
    return pd.DataFrame(res["data"], columns=res["columns"])

## 4. A first call, looked at raw

One quote from `yfinance_data.py`. Before converting anything, look at what came back: the top-level keys, the metadata, and the column types. The single row is then shown turned on its side, one field per line.

In [ ]:
res = run(["yfinance_data.py", "quote", "AAPL"])
print("top-level keys:", list(res))
if res["ok"]:
    meta = res["meta"]
    print(f"source: {meta['source']} | command: {meta['command']} | rows: {meta['count']} | as of: {meta['as_of']}")
    print("types of the first columns:", dict(list(res["types"].items())[:6]))
    raw_quote = pd.DataFrame(res["data"]).T.rename(columns={0: "AAPL"})
else:
    print(f"The quote failed [{res['code']}]: {res['error']}")
    raw_quote = None
raw_quote

## 5. A quote table for several symbols

`yfinance_data.py batch_quotes` takes any number of symbols and returns one row each. We keep the most useful columns and index the table by symbol.

In [ ]:
quotes = fetch("yfinance_data.py", "batch_quotes", *QUOTE_SYMBOLS)
if quotes is not None:
    wanted = ["symbol", "name", "price", "change", "change_percent", "volume", "currency", "market_date"]
    quotes = quotes[[c for c in wanted if c in quotes.columns]].set_index("symbol")
quotes

## 6. Price history into a DataFrame and a chart

`yfinance_data.py historical SYMBOL START END` returns daily rows with the columns `symbol, timestamp, open, high, low, close, volume, adj_close`. The `timestamp` is Unix time in seconds, so `pd.to_datetime(..., unit="s")` turns it into dates for the index.

In [ ]:
end = dt.date.today()
start = end - dt.timedelta(days=HISTORY_DAYS)
hist = fetch("yfinance_data.py", "historical", HISTORY_SYMBOL, start.isoformat(), end.isoformat())

if hist is not None:
    hist["date"] = pd.to_datetime(hist["timestamp"], unit="s").dt.normalize()
    hist = hist.set_index("date").drop(columns=["symbol", "timestamp"])

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hist["close"], label=f"{HISTORY_SYMBOL} close")
    ax.plot(hist["close"].rolling(20).mean(), label="20-day average")
    ax.set_title(f"{HISTORY_SYMBOL}: last {HISTORY_DAYS} days from yfinance_data.py historical")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")
    ax.legend(loc="upper left")
    plt.tight_layout()
hist.tail(10).round(2) if hist is not None else None

## 7. Beyond Yahoo: FX rates, no key needed

`exchangerate_data.py multi BASE T1,T2,...` returns the latest rates from open.er-api.com. The connector answers with `{currency: rate}`; the envelope turns that into `key` / `value` rows and keeps the base currency and update time in `meta["extra"]`.

In [ ]:
fx_res = run(["exchangerate_data.py", "multi", FX_BASE, ",".join(FX_TARGETS)])
if fx_res["ok"]:
    extra = fx_res["meta"]["extra"]
    print(f"Base {extra.get('base', FX_BASE)}, rates updated {extra.get('date', 'n/a')}")
    per_base = f"Units per 1 {FX_BASE}"
    fx = pd.DataFrame(fx_res["data"]).rename(columns={"key": "Currency", "value": per_base})
    fx = fx.set_index("Currency")
    fx[f"{FX_BASE} per 1 unit"] = 1 / fx[per_base]
else:
    print(f"FX rates unavailable [{fx_res['code']}]: {fx_res['error']}")
    fx = None
fx.round(4) if fx is not None else None

## 8. Crypto from CoinPaprika, also keyless

`coinpaprika_data.py tickers N` lists the top N crypto assets by market cap. Each row carries a nested `quotes` object (the envelope's `types` call it `object`); `pd.json_normalize` spreads it into ordinary columns such as `quotes.USD.price`.

In [ ]:
coins = fetch("coinpaprika_data.py", "tickers", str(TOP_COINS))
if coins is not None:
    coins = pd.json_normalize(coins.to_dict("records"))     # flatten the nested 'quotes' column
    keep = {"rank": "Rank", "name": "Name", "symbol": "Symbol",
            "quotes.USD.price": "Price (USD)",
            "quotes.USD.market_cap": "Market cap (USD bn)",
            "quotes.USD.percent_change_24h": "24h change (%)",
            "quotes.USD.percent_change_7d": "7d change (%)"}
    coins = coins[[c for c in keep if c in coins.columns]].rename(columns=keep).set_index("Rank")
    if "Market cap (USD bn)" in coins:
        coins["Market cap (USD bn)"] /= 1e9
coins.round(2) if coins is not None else None

## 9. When a call fails

`run_enveloped` (and so `run`) never raises for a failed fetch. It returns `ok: False` with a plain `error` message and a `code` you can branch on: `missing_key`, `auth_failed`, `rate_limited` (with `retry_after_s`), `upstream_down`, `bad_args`, `no_data`, `not_supported` or `unknown`. For a missing key it also gives `env_key` (the setting to fill in) and `docs_url` (where to get a key).

Two demonstrations: FRED needs a free API key, and to make the demo the same for everyone the cell hides `FRED_API_KEY` from this one call even if you have saved it; the second call simply leaves out the required dates.

In [ ]:
saved_key = os.environ.pop("FRED_API_KEY", None)      # hide a saved key for this demo only
try:
    fred = run(["fred_data.py", "series", "GDP"])
finally:
    if saved_key is not None:
        os.environ["FRED_API_KEY"] = saved_key
bad = run(["yfinance_data.py", "historical", "MSFT"])   # start/end dates missing

for label, res in [("fred_data.py series GDP", fred), ("yfinance_data.py historical MSFT", bad)]:
    if res["ok"]:
        print(f"{label}: worked, {res['meta']['count']} rows")
    elif res["code"] == "missing_key":
        print(f"{label}: needs a key - save {res.get('env_key')} in Settings > Credentials "
              f"(get one at {res.get('docs_url')})")
    else:
        print(f"{label}: failed [{res['code']}] {res['error']}")

pd.DataFrame([fred, bad], index=["fred_data.py series GDP", "yfinance_data.py historical MSFT"])

## 10. Which connectors need no key?

The catalogue `data_connectors.json` describes every connector: its script, commands, category and whether it needs a key. It lives in the terminal's `resources` folder, next to `scripts`. If the cell cannot find it (some installed layouts put it elsewhere), it tells you how to point at it by hand.

In [ ]:
candidates = [os.environ.get("FINCEPT_CONNECTOR_CATALOG", ""),
              os.path.join(os.path.dirname(fc.__file__), "..", "resources", "data_connectors.json")]
path = next((p for p in candidates if p and os.path.isfile(p)), None)

if path is None:
    print("data_connectors.json was not found next to the scripts folder. In an installed terminal it\n"
          "is in the 'resources' folder of the install directory: run\n"
          "    os.environ['FINCEPT_CONNECTOR_CATALOG'] = r'<full path to data_connectors.json>'\n"
          "in a cell and then run this cell again.")
    catalog = None
else:
    with open(path, encoding="utf-8") as fh:
        catalog = pd.DataFrame(json.load(fh)["connectors"])
    needs_key = catalog["key_required"].fillna(False).astype(bool)
    broken = catalog.get("broken", pd.Series(None, index=catalog.index, dtype=object)).notna()
    catalog["access"] = needs_key.map({True: "API key", False: "No key"})
    print(f"{len(catalog)} connectors; {int((~needs_key & ~broken).sum())} work without an API key.")
    display(pd.crosstab(catalog["category"], catalog["access"]))

A few keyless connectors per category, with their first commands. Any of them works with `fetch(script, command, *args)`; the `args_hint` of each command in the catalogue says which arguments it takes.

In [ ]:
if catalog is not None:
    usable = catalog[~needs_key & ~broken].copy()
    usable["commands"] = usable["commands"].map(lambda cs: ", ".join(c["name"] for c in cs[:4]))
    keyless = (usable.sort_values(["category", "title"])
               .groupby("category").head(3)
               [["category", "title", "script", "default_command", "commands"]]
               .set_index("category"))
else:
    keyless = None
keyless

## Try next

- Daily crypto candles: `fetch("coinpaprika_data.py", "ohlcv", "btc-bitcoin")`.
- Historical FX on a given day: `fetch("exchangerate_data.py", "historical", "2025-01-02", "EUR")`.
- Company fundamentals: `fetch("yfinance_data.py", "financial_ratios", "AAPL")`.
- Save a key (for example FRED) in **Settings > Credentials**, then re-run section 9 without the `os.environ.pop` line.

---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*